# 

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input/datasets/raneemidris/normal-gait"):
    print(root)
    print(files)

In [13]:
import shutil
import os

src = "/kaggle/input/datasets/raneemidris/normal/mpi_inf_3dhp/mpi_inf_3dhp"
dst = "/kaggle/working/mpi_inf_3dhp/"

shutil.copytree(src, dst, dirs_exist_ok=True)

'/kaggle/working/mpi_inf_3dhp_data/'

In [3]:
%cd /kaggle/working/mpi_inf_3dhp/

/kaggle/working/mpi_inf_3dhp


In [4]:
# !sed -i 's/ready_to_download=0/ready_to_download=1/' conf.ig

In [5]:
!sed -i 's|destination=.*|destination="/kaggle/working/mpi_3dhp_data"|' conf.ig

In [ ]:
!bash get_dataset.sh

Reading configuration from ./config.....
Download destination set to /kaggle/working/mpi_3dhp_data 
URL transformed to HTTPS due to an HSTS policy
--2026-06-09 13:35:21--  https://gvv.mpi-inf.mpg.de/3dhp-dataset/S1/Seq1/annot.mat
Resolving gvv.mpi-inf.mpg.de (gvv.mpi-inf.mpg.de)... 139.19.206.39
Connecting to gvv.mpi-inf.mpg.de (gvv.mpi-inf.mpg.de)|139.19.206.39|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://vcai.mpi-inf.mpg.de/3dhp-dataset/S1/Seq1/annot.mat [following]
--2026-06-09 13:35:22--  https://vcai.mpi-inf.mpg.de/3dhp-dataset/S1/Seq1/annot.mat
Resolving vcai.mpi-inf.mpg.de (vcai.mpi-inf.mpg.de)... 139.19.206.192
Connecting to vcai.mpi-inf.mpg.de (vcai.mpi-inf.mpg.de)|139.19.206.192|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 105891060 (101M)
Saving to: ‘annot.mat’

annot.mat           100%[===================>] 100.99M  29.4MB/s    in 3.9s    

2026-06-09 13:35:26 (26.2 MB/s) - ‘annot.mat’ saved 

In [ ]:
import os
imageSequence_folder = "/kaggle/working/mpi_3dhp_data/S1/Seq1/imageSequence"
print(os.listdir(imageSequence_folder))

hip adduction and abduction

saving keypoints

In [14]:
!pip install mediapipe

In [15]:
# !pip install mediapipe opencv-python numpy

import os
import cv2
import csv
import glob
import numpy as np
import mediapipe as mp

# تهيئة أداة MediaPipe Pose لتعقب المفاصل
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

print("Environment and MediaPipe Pose are successfully initialized!")

Environment and MediaPipe Pose are successfully initialized!


W0000 00:00:1781013454.601622     633 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781013454.653073     633 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [18]:
# المسار الأساسي للبيانات داخل كاجل كما حددتيه
base_path = "/kaggle/working/mpi_inf_3dhp_data"

# تحديد مسارات المخرجات وفصلها
train_output_folder = "/kaggle/working/keypoints/train"
val_output_folder   = "/kaggle/working/keypoints/validation"

os.makedirs(train_output_folder, exist_ok=True)
os.makedirs(val_output_folder, exist_ok=True)

# تقسيم الأشخاص بناءً على طلبكِ
train_subjects = ["S1", "S2", "S3", "S4", "S5", "S6"]
val_subjects   = ["S7", "S8"]
sequences      = ["Seq1", "Seq2"]

# مصفوفات لتجميع المجلدات المستهدفة مع تحديد نوعها (Train أو Val)
target_folders = []

# تجميع مجلدات التدريب (S1 - S6)
for s in train_subjects:
    for seq in sequences:
        path = f"{base_path}/{s}/{seq}/imageSequence"
        if os.path.exists(path):
            target_folders.append((s, seq, path, "train"))

# تجميع مجلدات التحقق (S7 - S8)
for s in val_subjects:
    for seq in sequences:
        path = f"{base_path}/{s}/{seq}/imageSequence"
        if os.path.exists(path):
            target_folders.append((s, seq, path, "val"))

print(f"Total folders found to process: {len(target_folders)}")

Total folders found to process: 16


In [19]:
print("Starting main processing pipeline...")

for subj, seq, folder_path, split_type in target_folders:
    # جلب جميع فيديوهات الـ .avi داخل المجلد الحالي
    video_files = glob.glob(os.path.join(folder_path, "*.avi"))
    
    # تحديد مجلد الحفظ بناءً على نوع الـ Split (Train أو Val)
    current_output_dir = train_output_folder if split_type == "train" else val_output_folder
    
    for video_path in video_files:
        video_name = os.path.basename(video_path).split('.')[0]
        print(f"[{split_type.upper()}] Processing: Subject {subj} | {seq} | Video: {video_name}")
        
        csv_filename = f"{subj}_{seq}_{video_name}_keypoints.csv"
        csv_output_path = os.path.join(current_output_dir, csv_filename)
        
        cap = cv2.VideoCapture(video_path)
        
        with open(csv_output_path, mode='w', newline='') as f:
            writer = csv.writer(f)
            
            # كتابة الـ Headers للـ 48 ميزة المطلوبة بدقة للموديل
            headers = [f"Feature_{i}" for i in range(1, 49)]
            writer.writerow(headers)
            
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break
                
                rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                results = pose.process(rgb_frame)
                
                if results.pose_landmarks:
                    landmarks = results.pose_landmarks.landmark
                    
                    # استخراج إحداثيات المفاصل الأساسية (X, Y, Z)
                    l_shoulder = np.array([landmarks[11].x, landmarks[11].y, landmarks[11].z])
                    r_shoulder = np.array([landmarks[12].x, landmarks[12].y, landmarks[12].z])
                    l_hip = np.array([landmarks[23].x, landmarks[23].y, landmarks[23].z])
                    r_hip = np.array([landmarks[24].x, landmarks[24].y, landmarks[24].z])
                    l_knee = np.array([landmarks[25].x, landmarks[25].y, landmarks[25].z])
                    r_knee = np.array([landmarks[26].x, landmarks[26].y, landmarks[26].z])
                    l_ankle = np.array([landmarks[27].x, landmarks[27].y, landmarks[27].z])
                    r_ankle = np.array([landmarks[28].x, landmarks[28].y, landmarks[28].z])
                    
                    # 1. جعل مركز الحوض (Mid-Hip) هو نقطة الصفر (0,0,0) لكل فريم
                    mid_hip = (l_hip + r_hip) / 2.0
                    
                    # 2. حساب مقياس الجسم (عرض الكتفين) لتوحيد المسافات (Scale Invariance)
                    shoulder_width = np.linalg.norm(l_shoulder - r_shoulder)
                    if shoulder_width == 0: 
                        shoulder_width = 1.0
                    
                    # تطبيق التطبيع (Normalization)
                    norm_joints = []
                    for joint in [l_hip, r_hip, l_shoulder, r_shoulder, l_knee, r_knee, l_ankle, r_ankle]:
                        normalized = (joint - mid_hip) / shoulder_width
                        norm_joints.extend(normalized)
                    
                    # 3. تعبئة الأبعاد بـ الأصفار لتتطابق تماماً مع حجم الـ 48 ميزة المحددة في مشروعك
                    frame_features = np.zeros(48)
                    frame_features[:len(norm_joints)] = norm_joints
                    
                    writer.writerow(frame_features)
                    
        cap.release()
        print(f"-> Saved successfully to: {split_type}/{csv_filename}")

print("\n[SUCCESS] All keypoints extracted, normalized, and split into Train and Validation subsets!")

Starting main processing pipeline...
[TRAIN] Processing: Subject S1 | Seq1 | Video: video_0
-> Saved successfully to: train/S1_Seq1_video_0_keypoints.csv
[TRAIN] Processing: Subject S1 | Seq1 | Video: video_2
-> Saved successfully to: train/S1_Seq1_video_2_keypoints.csv
[TRAIN] Processing: Subject S1 | Seq1 | Video: video_5


KeyboardInterrupt: 

In [ ]:
mp_pose = mp.solutions.pose

joints = [
    mp_pose.PoseLandmark.LEFT_SHOULDER,
    mp_pose.PoseLandmark.RIGHT_SHOULDER,
    mp_pose.PoseLandmark.LEFT_HIP,
    mp_pose.PoseLandmark.RIGHT_HIP,
    mp_pose.PoseLandmark.LEFT_KNEE,
    mp_pose.PoseLandmark.RIGHT_KNEE,
    mp_pose.PoseLandmark.LEFT_ANKLE,
    mp_pose.PoseLandmark.RIGHT_ANKLE,
]

In [ ]:

import random

train_folder = os.path.join(output_folder, "train")
val_folder = os.path.join(output_folder, "val")

os.makedirs(train_folder, exist_ok=True)
os.makedirs(val_folder, exist_ok=True)

train_csv = os.path.join(train_folder, "keypoints.csv")
val_csv = os.path.join(val_folder, "keypoints.csv")

all_videos = []
for folder in video_folders:
    avi_files = sorted([f for f in os.listdir(folder) if f.lower().endswith(".avi")])
    for video_file in avi_files:
        all_videos.append((folder, video_file))

random.seed(42)
random.shuffle(all_videos)

split_idx = int(0.8 * len(all_videos))
train_videos = set(all_videos[:split_idx])

train_f = open(train_csv, "w", newline="")
val_f = open(val_csv, "w", newline="")

train_writer = csv.writer(train_f)
val_writer = csv.writer(val_f)

header = ["subject", "sequence", "video", "frame"]
for j in joints:
    name = j.name.lower()
    header += [f"{name}_x", f"{name}_y", f"{name}_z"]

train_writer.writerow(header)
val_writer.writerow(header)

for folder in video_folders:

    avi_files = sorted([f for f in os.listdir(folder) if f.lower().endswith(".avi")])

    print(f"\nFolder: {folder}")
    print(f"Found {len(avi_files)} videos")

    for video_file in avi_files:

        video_path = os.path.join(folder, video_file)

        subject = os.path.basename(os.path.dirname(os.path.dirname(folder)))
        seq = os.path.basename(os.path.dirname(folder))

        writer = train_writer if (folder, video_file) in train_videos else val_writer

        cap = cv2.VideoCapture(video_path)
        frame_count = 0

        while cap.isOpened():

            ret, frame = cap.read()
            if not ret:
                break

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = pose.process(rgb)

            row = [subject, seq, video_file, frame_count]

            if results.pose_landmarks:
                for joint in joints:
                    lm = results.pose_landmarks.landmark[joint]
                    row += [lm.x, lm.y, lm.z]
            else:
                row += [np.nan] * (len(joints) * 3)

            writer.writerow(row)
            frame_count += 1

        cap.release()

train_f.close()
val_f.close()

print(f"Saved train keypoints to: {train_csv}")
print(f"Saved val keypoints to: {val_csv}")


In [ ]:
pip install opencv-python mediapipe numpy pandas matplotlib

In [ ]:
import cv2
import mediapipe as mp
from mediapipe.python.solutions import pose as mp_pose
from mediapipe.python.solutions import drawing_utils as mp_drawing
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def calculate_angle(a, b, c):
    """
    Calculates the angle (in degrees) between three points.
    b is the vertex point.
    """
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)
    
    ba = a - b
    bc = c - b
    
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    cosine_angle = np.clip(cosine_angle, -1.0, 1.0)
    
    angle = np.degrees(np.arccos(cosine_angle))
    return angle

def process_gait_video(input_video_path, output_video_path, output_csv_path, output_plot_path):
    # Initialize MediaPipe Pose using the directly imported modules
    pose = mp_pose.Pose(
        static_image_mode=False,
        model_complexity=1,
        smooth_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    )
    
    # Define the 8 key joints used in your notebook
    joints = [
        mp_pose.PoseLandmark.LEFT_SHOULDER,
        mp_pose.PoseLandmark.RIGHT_SHOULDER,
        mp_pose.PoseLandmark.LEFT_HIP,
        mp_pose.PoseLandmark.RIGHT_HIP,
        mp_pose.PoseLandmark.LEFT_KNEE,
        mp_pose.PoseLandmark.RIGHT_KNEE,
        mp_pose.PoseLandmark.LEFT_ANKLE,
        mp_pose.PoseLandmark.RIGHT_ANKLE
    ]
    
    # Open input video
    cap = cv2.VideoCapture(input_video_path)
    if not cap.isOpened():
        print(f"Error: Could not open video file {input_video_path}")
        return
        
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    if fps == 0: 
        fps = 30  # Fallback
        
    # Setup Video Writer for visualized output
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))
    
    # Data tracking collections
    data_records = []
    frames = []
    left_hip_angles = []
    right_hip_angles = []
    
    frame_count = 0
    print("Processing video frames...")
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        frame_count += 1
        # Convert the frame to RGB for MediaPipe processing
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(rgb)
        
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark
            
            # Dictionary to collect all keypoint fields for the current frame
            frame_data = {"frame": frame_count}
            
            # 1. Extract raw coordinates (x, y, z) for the 8 joints
            for joint in joints:
                lm = landmarks[joint.value]
                name = joint.name.lower()
                frame_data[f"{name}_x"] = lm.x
                frame_data[f"{name}_y"] = lm.y
                frame_data[f"{name}_z"] = lm.z
                
            # 2. Get coordinates for Hip Angle calculation
            left_hip = [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y]
            right_hip = [landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value].y]
            left_knee = [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y]
            right_knee = [landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value].y]
            
            # Calculate Mid Hip Center (Pelvis Center)
            mid_hip = [
                (left_hip[0] + right_hip[0]) / 2,
                (left_hip[1] + right_hip[1]) / 2
            ]
            
            # 3. Compute True Hip Adduction / Abduction Angles
            left_angle = calculate_angle(mid_hip, left_hip, left_knee)
            right_angle = calculate_angle(mid_hip, right_hip, right_knee)
            
            # Save metrics to record lists
            frames.append(frame_count)
            left_hip_angles.append(left_angle)
            right_hip_angles.append(right_angle)
            
            frame_data["left_hip_angle"] = left_angle
            frame_data["right_hip_angle"] = right_angle
            data_records.append(frame_data)
            
            # --- VISUALIZATION OVERLAYS ---
            # Draw standard skeleton connections
            mp_drawing.draw_landmarks(frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
            
            # Convert normalized landmarks to frame pixel coordinates
            mid_hip_px = (int(mid_hip[0] * width), int(mid_hip[1] * height))
            left_hip_px = (int(left_hip[0] * width), int(left_hip[1] * height))
            right_hip_px = (int(right_hip[0] * width), int(right_hip[1] * height))
            left_knee_px = (int(left_knee[0] * width), int(left_knee[1] * height))
            right_knee_px = (int(right_knee[0] * width), int(right_knee[1] * height))
            
            # Highlight our 8 targeted keypoint circles
            for joint in joints:
                lm = landmarks[joint.value]
                cv2.circle(frame, (int(lm.x * width), int(lm.y * height)), 7, (0, 255, 0), -1)
                
            # Draw custom gait analysis lines (Mid-hip connection & Angles)
            cv2.circle(frame, mid_hip_px, 9, (0, 255, 255), -1) # Yellow mid-hip point
            cv2.line(frame, mid_hip_px, left_hip_px, (0, 255, 255), 2)
            cv2.line(frame, left_hip_px, left_knee_px, (0, 255, 0), 2)
            cv2.line(frame, mid_hip_px, right_hip_px, (0, 255, 255), 2)
            cv2.line(frame, right_hip_px, right_knee_px, (255, 0, 0), 2)
            
            # Add dynamic text tracking angle values
            cv2.putText(frame, f'L Hip: {int(left_angle)}', left_hip_px, cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f'R Hip: {int(right_angle)}', right_hip_px, cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2, cv2.LINE_AA)
        
        else:
            # Handle frames where no person or pose is detected seamlessly
            frame_data = {"frame": frame_count}
            for joint in joints:
                name = joint.name.lower()
                frame_data[f"{name}_x"] = np.nan
                frame_data[f"{name}_y"] = np.nan
                frame_data[f"{name}_z"] = np.nan
            frame_data["left_hip_angle"] = np.nan
            frame_data["right_hip_angle"] = np.nan
            data_records.append(frame_data)
            
        out.write(frame)
        
    cap.release()
    out.write(frame)
    out.release()
    pose.close()
    print("Video processing finished.")
    
    # --- EXPORT DATA TO CSV ---
    df = pd.DataFrame(data_records)
    df.to_csv(output_csv_path, index=False)
    print(f" Saved structural keypoints to: {output_csv_path}")
    print(f" Visualized skeleton video saved to: {output_video_path}")
    
    # --- VISUALIZE TIME-SERIES PLOT ---
    if frames:
        plt.figure(figsize=(12, 6))
        plt.plot(frames, left_hip_angles, label='Left Hip Angle', color='#2ca02c', linewidth=2)
        plt.plot(frames, right_hip_angles, label='Right Hip Angle', color='#1f77b4', linewidth=2)
        plt.xlabel("Frame Number")
        plt.ylabel("Angle (Degrees)")
        plt.title("Hip Adduction / Abduction Gait Angle Visualizer")
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.tight_layout()
        plt.savefig(output_plot_path, dpi=300)
        plt.close()
        print(f" Analytics graph plot saved to: {output_plot_path}")

# ==========================================
# EXECUTION ROUTINE
# ==========================================
if __name__ == "__main__":
    # Ensure you replace this with the path to your actual video file
    input_video = "/kaggle/input/datasets/raneemidris/abnormal-gait/IMG_2325.MP4" 
    
    output_video = "visualized_skeleton_output.mp4"
    output_csv = "extracted_gait_keypoints.csv"
    output_plot = "gait_angle_trends.png"
    
    process_gait_video(input_video, output_video, output_csv, output_plot)

In [ ]:
!pip install mediapipe==0.10.14 -q